# cAPTure: packet and history MLP development screening

This GPU notebook trains two matched feed-forward baselines on the existing cAPTure development folds:

- `MLP-P` uses the same 103 fold-preprocessed packet features as XGB-P.
- `MLP-History` adds only the six causal features computed from the six complete preceding five-second windows.

Both variants reuse the frozen preprocessing, scenario/class weights, fold assignments, seed, and out-of-fold evaluation protocol. Training uses ten fixed epochs and never consults the outer validation scenarios for checkpoint selection. The primary operational decision time remains the five-second window close so the results are directly comparable with the XGB tables. Packet-arrival timing remains available for a later sensitivity audit. Held-out author-train and final-test scenarios are not read.


## 1. Prepare the Colab GPU environment

Select a GPU runtime before running this section. The notebook writes final models and OOF scores to Drive and uses `/content` only for temporary training matrices.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import gc
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_WORK_ROOT = Path("/content/capture_mlp_work")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

required_files = [
    "code/python/requirements-capture-mlp.txt",
    "code/python/utils/capture_mlp.py",
    "code/python/utils/models.py",
    "configs/capture_experiment_v1.yaml",
]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
import torch
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA-enabled Colab runtime before training the MLPs.")
print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("PyTorch version:", torch.__version__)
print("CUDA device:", torch.cuda.get_device_name(0))


## 2. Bind the completed development artifacts

Set `MLP_RUN_ID` only when resuming an existing MLP run. The four fold/variant directories are immutable and are verified before reuse. The history model requires the completed context run used by XGB-P+T.


In [ ]:
from utils.capture_mlp import (
    run_capture_mlp_fold,
    run_capture_mlp_operational_evaluation,
    summarize_capture_mlp_oof,
    validate_capture_mlp_fold_run,
    validate_capture_mlp_operational_evaluation,
)

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
PREPROCESSING_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml"
PREPARED_RUN_DIR = (
    DRIVE_ROOT / "prepared_runs/20260917T235058_827743Z_prepare_full_dev"
)
PREPROCESSING_AUDIT_DIR = (
    DRIVE_ROOT / "preprocessing_runs/20260919T004143_161396Z_preprocessing"
)
CONTEXT_RUN_ID = "20260919T201909_754628Z_xgb_p_t"
CONTEXT_DIR = DRIVE_ROOT / "xgb_p_t_context_runs" / CONTEXT_RUN_ID

MLP_RUN_ID = None  # Set an existing ID only when resuming.
if MLP_RUN_ID is None:
    MLP_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_mlp_screen"
    )
MLP_RUN_DIR = DRIVE_ROOT / "mlp_runs" / MLP_RUN_ID
MLP_OPERATIONAL_DIR = (
    DRIVE_ROOT / "mlp_operational_runs" / f"{MLP_RUN_ID}_operational"
)

for required_path in (
    MANIFEST_PATH,
    PACKET_SCHEMA_PATH,
    PREPROCESSING_SCHEMA_PATH,
    PREPARED_RUN_DIR,
    PREPROCESSING_AUDIT_DIR,
    CONTEXT_DIR,
):
    if not required_path.exists():
        raise FileNotFoundError(f"Required input is missing: {required_path}")

print("MLP run ID:", MLP_RUN_ID)
print("MLP output:", MLP_RUN_DIR)
print("Operational output:", MLP_OPERATIONAL_DIR)


## 3. Define resumable fold execution

Each call either validates a completed immutable fold or trains it from scratch. Training materializes the fold matrix on local Colab storage, copies only the checkpoint, reports, scaler, and OOF predictions to Drive, and removes the temporary matrix afterward.


In [ ]:
def train_or_verify(variant_name, fold):
    output_dir = MLP_RUN_DIR / variant_name / f"fold_{fold}"
    if output_dir.exists():
        print(f"Validating existing {variant_name} fold {fold}...")
        report = validate_capture_mlp_fold_run(output_dir, fold, variant_name)
    else:
        report = run_capture_mlp_fold(
            manifest_path=MANIFEST_PATH,
            packet_schema_path=PACKET_SCHEMA_PATH,
            preprocessing_schema_path=PREPROCESSING_SCHEMA_PATH,
            prepared_run_dir=PREPARED_RUN_DIR,
            preprocessing_audit_dir=PREPROCESSING_AUDIT_DIR,
            context_dir=CONTEXT_DIR,
            output_dir=output_dir,
            local_work_root=LOCAL_WORK_ROOT,
            fold=fold,
            variant_name=variant_name,
        )
    gc.collect()
    torch.cuda.empty_cache()
    print(
        f"{variant_name} fold {fold}: "
        f"macro ROC-AUC={report['fold_macro_packet_roc_auc']:.6f}"
    )
    return report


## 4. Train MLP-P

This is the direct neural counterpart of XGB-P. It operates independently on each packet and receives no window or history summaries.


In [ ]:
packet_fold_a = train_or_verify("packet", "A")


In [ ]:
packet_fold_b = train_or_verify("packet", "B")
packet_summary = summarize_capture_mlp_oof(MLP_RUN_DIR, "packet")
display(pd.DataFrame.from_dict(packet_summary["scenario_metrics"], orient="index"))
print(
    "MLP-P hierarchical macro OOF packet ROC-AUC:",
    packet_summary["hierarchical_macro_oof_packet_roc_auc"],
)


## 5. Train MLP-History

This model has the same architecture and training policy as MLP-P. Its only additional inputs are the six summaries from the preceding 30 seconds; the current five-second window is excluded.


In [ ]:
history_fold_a = train_or_verify("history", "A")


In [ ]:
history_fold_b = train_or_verify("history", "B")
history_summary = summarize_capture_mlp_oof(MLP_RUN_DIR, "history")
display(pd.DataFrame.from_dict(history_summary["scenario_metrics"], orient="index"))
print(
    "MLP-History hierarchical macro OOF packet ROC-AUC:",
    history_summary["hierarchical_macro_oof_packet_roc_auc"],
)


## 6. Compare threshold-free packet ranking

The hierarchical macro first averages scenarios within each fold and then averages the two folds. The delta isolates the effect of the six history features for this fixed MLP configuration.


In [ ]:
mlp_summaries = {
    "mlp_p": packet_summary,
    "mlp_history": history_summary,
}
ranking_rows = []
for model_name, summary in mlp_summaries.items():
    ranking_rows.append({
        "model": model_name,
        "hierarchical_macro_oof_packet_roc_auc": (
            summary["hierarchical_macro_oof_packet_roc_auc"]
        ),
        "hierarchical_macro_oof_packet_average_precision": (
            summary["hierarchical_macro_oof_packet_pr_auc_diagnostic"]
        ),
    })
ranking_table = pd.DataFrame(ranking_rows).set_index("model")
display(ranking_table)
print(
    "History minus packet ROC-AUC:",
    ranking_table.loc["mlp_history", "hierarchical_macro_oof_packet_roc_auc"]
    - ranking_table.loc["mlp_p", "hierarchical_macro_oof_packet_roc_auc"],
)


## 7. Select operational thresholds and evaluate OOF alerts

This section applies the existing frozen budgets: one false-alert window per hour, one per 12 hours, and one per five minutes. Threshold selection uses only development OOF predictions. Alert availability is aligned to the five-second window close for direct comparison with XGB.


In [ ]:
if MLP_OPERATIONAL_DIR.exists():
    operational_report = validate_capture_mlp_operational_evaluation(
        manifest_path=MANIFEST_PATH,
        mlp_run_dir=MLP_RUN_DIR,
        output_dir=MLP_OPERATIONAL_DIR,
    )
else:
    operational_report = run_capture_mlp_operational_evaluation(
        manifest_path=MANIFEST_PATH,
        mlp_run_dir=MLP_RUN_DIR,
        output_dir=MLP_OPERATIONAL_DIR,
    )

operational_rows = []
for model_name, model_report in operational_report["models"].items():
    for budget_name in operational_report["budget_order"]:
        threshold_report = model_report["thresholds"][budget_name]
        metrics = model_report["budgets"][budget_name]["hierarchical_macro"]
        operational_rows.append({
            "model": model_name,
            "budget": budget_name,
            "threshold": threshold_report["threshold"],
            "worst_fold_false_alert_windows_per_hour": (
                threshold_report["worst_fold_false_alert_windows_per_hour"]
            ),
            **metrics,
        })
operational_table = pd.DataFrame(operational_rows).set_index(["model", "budget"])
display(operational_table)


## 8. Report classical packet-classification metrics

These metrics are computed over packets at the selected one-per-hour operational threshold. ROC-AUC and average precision remain threshold-free. Scenario rows expose heterogeneity; the summary uses the same fold-then-fold hierarchical macro as the rest of the study.


In [ ]:
def safe_ratio(numerator, denominator):
    return numerator / denominator if denominator else float("nan")


def f_beta(precision, recall, beta):
    beta_squared = beta ** 2
    denominator = beta_squared * precision + recall
    return (
        (1 + beta_squared) * precision * recall / denominator
        if denominator else float("nan")
    )


classic_rows = []
primary_budget = "one_per_hour"
for model_name, model_report in operational_report["models"].items():
    variant_name = model_report["variant_name"]
    ranking_by_scenario = mlp_summaries[model_name]["scenario_metrics"]
    scenario_metrics = model_report["budgets"][primary_budget]["scenario_metrics"]
    for scenario, metrics in scenario_metrics.items():
        true_positives = metrics["packet_true_positives"]
        false_positives = metrics["packet_false_positives"]
        true_negatives = metrics["packet_true_negatives"]
        false_negatives = metrics["packet_false_negatives"]
        precision = safe_ratio(true_positives, true_positives + false_positives)
        recall = safe_ratio(true_positives, true_positives + false_negatives)
        classic_rows.append({
            "model": model_name,
            "scenario": scenario,
            "fold": metrics["fold"],
            "budget": primary_budget,
            "threshold": metrics["threshold"],
            "packet_precision": precision,
            "packet_recall": recall,
            "packet_f1": f_beta(precision, recall, 1),
            "packet_f2": f_beta(precision, recall, 2),
            "packet_roc_auc": ranking_by_scenario[scenario]["packet_roc_auc"],
            "packet_average_precision": ranking_by_scenario[scenario][
                "packet_pr_auc_diagnostic"
            ],
            "packet_false_positive_rate": safe_ratio(
                false_positives, false_positives + true_negatives
            ),
            "false_alert_windows_per_hour": metrics[
                "false_alert_windows_per_hour"
            ],
            "packet_true_positives": true_positives,
            "packet_false_positives": false_positives,
            "packet_true_negatives": true_negatives,
            "packet_false_negatives": false_negatives,
        })

classic_table = pd.DataFrame(classic_rows)
display(classic_table.set_index(["model", "scenario"]).sort_index())

summary_fields = [
    "packet_precision",
    "packet_recall",
    "packet_f1",
    "packet_f2",
    "packet_roc_auc",
    "packet_average_precision",
    "packet_false_positive_rate",
    "false_alert_windows_per_hour",
]
fold_means = (
    classic_table.groupby(["model", "fold"], as_index=False)[summary_fields].mean()
)
classic_hierarchical = (
    fold_means.groupby("model", as_index=True)[summary_fields].mean()
)
thresholds = {
    model_name: model_report["thresholds"][primary_budget]["threshold"]
    for model_name, model_report in operational_report["models"].items()
}
classic_hierarchical.insert(
    0,
    "threshold",
    [thresholds[model_name] for model_name in classic_hierarchical.index],
)
display(classic_hierarchical)


## 9. Interpretation boundary

This is a one-seed development screen of one frozen MLP configuration. It can answer whether a feed-forward nonlinear model benefits from the causal history fields under the current folds. It does not estimate seed variability, complete a neural hyperparameter search, or provide final generalization evidence. Keep Test1 and Test2 untouched until the model and graph protocol are frozen.
